In [3]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Loading our attrition processed data into a pandas dataframe
df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

# Visualizing top 5 records
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


### Exploratory Data Analysis
#### 1. Where is attrition concentrated?
- Which job roles, departments, and job levels exceed the 16.1% baseline, and by enough to matter given their headcount?
- Is attrition an early-tenure phenomenon, a mid-career phenomenon, or both?

In [4]:
# Checking the overall attrition rate in the dataset
attrition_rate = (df["Attrition"] == "Yes").mean()
print(f"Overall attrition rate: {attrition_rate:.1%}")

Overall attrition rate: 16.1%


In [10]:
# Defining our attrition rate baseline
BASELINE = attrition_rate

In [26]:
# Creating a reusable attrition sumamry function for our analysis
def attrition_summary(df, column, baseline=BASELINE):
    summary = (
        df.groupby(column)
          .agg(
              Headcount=("EmployeeNumber", "size"),
              AttritionCount=("Attrition", lambda x: (x == "Yes").sum())
          )
    )

    # Calculate attrition rate as a percentage
    summary["AttritionRate"] = (
        summary["AttritionCount"] / summary["Headcount"] * 100
    )

    # Difference from company-wide baseline, in percentage points
    summary["DifferenceFromBaseline_pp"] = (
        summary["AttritionRate"] - (baseline * 100)
    )

    # How many times higher/lower than the baseline
    summary["RelativeToBaseline"] = (
        summary["AttritionRate"] / (baseline * 100)
    )

    return summary.sort_values(
        "AttritionRate",
        ascending=False
    )

In [30]:
# Checking attrition summary by JobRole
job_role_summary = attrition_summary(df, "JobRole")

job_role_summary

,Headcount,AttritionCount,AttritionRate,DifferenceFromBaseline_pp,RelativeToBaseline
JobRole,,,,,
Sales Representative,83,33,39.759036,23.636587,2.466067
Laboratory Technician,259,62,23.938224,7.815775,1.484776
Human Resources,52,12,23.076923,6.954474,1.431353
Sales Executive,326,57,17.484663,1.362214,1.084492
Research Scientist,292,47,16.095890,-0.026559,0.998353
Manufacturing Director,145,10,6.896552,-9.225897,0.427761
Healthcare Representative,131,9,6.870229,-9.252220,0.426128
Manager,102,5,4.901961,-11.220488,0.304046
Research Director,80,2,2.500000,-13.622449,0.155063


In [31]:
# Checking attrition summary by Department
department_summary = attrition_summary(df, "Department")

department_summary

,Headcount,AttritionCount,AttritionRate,DifferenceFromBaseline_pp,RelativeToBaseline
Department,,,,,
Sales,446,92,20.627803,4.505354,1.279446
Human Resources,63,12,19.047619,2.925170,1.181435
Research & Development,961,133,13.839750,-2.282699,0.858415


In [32]:
# Checking attrition summary by JobLevel
job_level_summary = attrition_summary(df, "JobLevel")

job_level_summary

,Headcount,AttritionCount,AttritionRate,DifferenceFromBaseline_pp,RelativeToBaseline
JobLevel,,,,,
1,543,143,26.335175,10.212726,1.633448
3,218,32,14.678899,-1.443550,0.910463
2,534,52,9.737828,-6.384621,0.603992
5,69,5,7.246377,-8.876072,0.449459
4,106,5,4.716981,-11.405468,0.292572


- Which job roles, departments, and job levels exceed the 16.1% baseline, and by enough to matter given their headcount?

In [43]:
# To better analyse the organizational impact of the JobRole attrition rate, we'll add a new attribute for Excess attrition
# Through Excess attrition we can analyse how many more employees left than we would expect if this group had the company-wide baseline?
# Expected Attrition = Headcount*BASELINE
# Excess Attrition=Actual Attrition−Expected Attrition

job_role_summary["ExpectedAttrition"] = (
    job_role_summary["Headcount"] * BASELINE
).round(2)

job_role_summary["ExcessAttrition"] = (
    job_role_summary["AttritionRate"] -
    job_role_summary["ExpectedAttrition"]
).round(2)

job_role_summary.sort_values(
    "ExcessAttrition",
    ascending=False
)

,Headcount,AttritionCount,AttritionRate,DifferenceFromBaseline_pp,RelativeToBaseline,ExpectedAttrition,ExcessAttrition
JobRole,,,,,,,
Sales Representative,83,33,39.759036,23.636587,2.466067,13.38,26.38
Human Resources,52,12,23.076923,6.954474,1.431353,8.38,14.70
Research Director,80,2,2.500000,-13.622449,0.155063,12.90,-10.40
Manager,102,5,4.901961,-11.220488,0.304046,16.44,-11.54
Healthcare Representative,131,9,6.870229,-9.252220,0.426128,21.12,-14.25
Manufacturing Director,145,10,6.896552,-9.225897,0.427761,23.38,-16.48
Laboratory Technician,259,62,23.938224,7.815775,1.484776,41.76,-17.82
Research Scientist,292,47,16.095890,-0.026559,0.998353,47.08,-30.98
Sales Executive,326,57,17.484663,1.362214,1.084492,52.56,-35.08


In [44]:
# Formatting JobRole summary for better readability
role_display = job_role_summary.copy()

role_display["AttritionRate"] = role_display["AttritionRate"].round(2)

role_display["DifferenceFromBaseline_pp"] = role_display["DifferenceFromBaseline_pp"].round(2)

role_display["RelativeToBaseline"] = role_display["RelativeToBaseline"].round(2)

role_display

,Headcount,AttritionCount,AttritionRate,DifferenceFromBaseline_pp,RelativeToBaseline,ExpectedAttrition,ExcessAttrition
JobRole,,,,,,,
Sales Representative,83,33,39.76,23.64,2.47,13.38,26.38
Laboratory Technician,259,62,23.94,7.82,1.48,41.76,-17.82
Human Resources,52,12,23.08,6.95,1.43,8.38,14.70
Sales Executive,326,57,17.48,1.36,1.08,52.56,-35.08
Research Scientist,292,47,16.10,-0.03,1.00,47.08,-30.98
Manufacturing Director,145,10,6.90,-9.23,0.43,23.38,-16.48
Healthcare Representative,131,9,6.87,-9.25,0.43,21.12,-14.25
Manager,102,5,4.90,-11.22,0.30,16.44,-11.54
Research Director,80,2,2.50,-13.62,0.16,12.90,-10.40
